# Event Streaming with Kafka & Spark Structured Streaming

## Objective

Earlier notebooks implemented batch-oriented ingestion using files stored in S3. That architecture is appropriate when data arrives as discrete files.

In this notebook, I explore event-driven processing for data that arrives continuously.

I use Apache Kafka as the event broker and Spark Structured Streaming as the processing engine. Synthetic purchase-like events are published individually to a Kafka topic, consumed by Spark, validated against an explicit schema, separated into trusted and quarantined streams, and deduplicated using event-time state.

The objective is to understand the architectural differences between batch ingestion and streaming ingestion while demonstrating Kafka partitions, offsets, Structured Streaming checkpoints, watermarks, event-time processing, and stateful deduplication.

The events in this notebook are synthetic and remain isolated from the original X5 warehouse and uplift-modeling data.

## 1. Streaming architecture

Kafka and Spark have separate responsibilities.

Kafka acts as the durable event log. Producers publish events to a topic, and Kafka assigns each event to a partition and offset.

Spark Structured Streaming consumes those events and applies the project's processing logic.

For this demonstration, the processing path is:

`producer → Kafka → Spark → validation → valid/quarantine → Parquet`

Each trusted event retains its Kafka partition and offset so its source position remains observable.

The streaming application also stores checkpoints. Checkpoints allow Spark to track which Kafka offsets have already been processed and preserve state required by streaming operations such as deduplication.

A 10-minute event-time watermark bounds the amount of state retained for event-ID deduplication. The watermark is based on when an event occurred rather than when Spark happened to receive it.

In [2]:
# ============================================================
# 1. Inspect Structured Streaming outputs
#
# These Parquet datasets were created by Spark.
# This cell does not rerun the streaming pipeline.
# ============================================================

from pathlib import Path

import pandas as pd


PROJECT_ROOT = next(
    path
    for path in [
        Path.cwd(),
        *Path.cwd().parents,
    ]
    if (
        path
        / "dbt"
        / "dbt_project.yml"
    ).exists()
)


VALID_PATH = (
    PROJECT_ROOT
    / "data"
    / "streaming"
    / "output"
    / "valid"
)

QUARANTINE_PATH = (
    PROJECT_ROOT
    / "data"
    / "streaming"
    / "output"
    / "quarantine"
)


valid_events = pd.read_parquet(
    VALID_PATH
)

quarantined_events = pd.read_parquet(
    QUARANTINE_PATH
)


print(
    f"Trusted events: "
    f"{len(valid_events):,}"
)

print(
    f"Quarantined events: "
    f"{len(quarantined_events):,}"
)


display(
    valid_events.sort_values(
        "event_id"
    )
)

display(
    quarantined_events[
        [
            "event_id",
            "rejection_reason",
            "kafka_partition",
            "kafka_offset",
        ]
    ].sort_values(
        "event_id"
    )
)

Trusted events: 4
Quarantined events: 2


,event_id,client_id,event_ts,amount,schema_version,kafka_partition,kafka_offset,kafka_timestamp
0,STREAM_A100,stream_customer_a,2026-09-23 17:00:00,25.50,1,1,0,2026-09-23 20:36:06.125
3,STREAM_B100,stream_customer_b,2026-09-23 17:01:00,12.00,1,2,0,2026-09-23 20:36:06.126
1,STREAM_C100,stream_customer_c,2026-09-23 17:02:00,48.25,1,1,1,2026-09-23 20:36:06.126
2,STREAM_D100,stream_customer_d,2026-09-23 17:03:00,17.75,1,2,1,2026-09-23 20:36:06.126


,event_id,rejection_reason,kafka_partition,kafka_offset
1,STREAM_F100,INVALID_AMOUNT,1,3
0,STREAM_G100,INVALID_EVENT_TIMESTAMP,2,2


In [3]:
# ============================================================
# 2. Kafka partition / offset provenance
#
# The same logical event stream can be distributed across
# multiple Kafka partitions.
#
# Offsets identify an event's position WITHIN a partition.
# They are not globally sequential across the whole topic.
# ============================================================

partition_summary = (
    valid_events
    .groupby(
        "kafka_partition"
    )
    .agg(
        events=(
            "event_id",
            "count",
        ),
        min_offset=(
            "kafka_offset",
            "min",
        ),
        max_offset=(
            "kafka_offset",
            "max",
        ),
    )
    .reset_index()
)

display(
    partition_summary
)

,kafka_partition,events,min_offset,max_offset
0,1,2,0,1
1,2,2,0,1


## 2. Streaming results

The base producer published seven Kafka messages.

Spark classified the stream into valid and invalid populations before writing the results to separate Parquet datasets.

### Initial processing

- Kafka messages published: **7**
- Trusted unique events: **4**
- Quarantined events: **2**
- Duplicate event retries suppressed: **1**
- Kafka topic partitions: **3**
- Event-time watermark: **10 minutes**

The invalid amount event was quarantined as `INVALID_AMOUNT`, while the invalid timestamp event was quarantined as `INVALID_EVENT_TIMESTAMP`.

The repeated `STREAM_A100` message did not create a second trusted event.

### Checkpoint verification

I reran the Spark streaming application without publishing any new Kafka messages.

- Trusted events before rerun: **4**
- Trusted events after rerun: **4**
- Quarantined events before rerun: **2**
- Quarantined events after rerun: **2**

The unchanged populations demonstrate that Spark resumed from its recorded streaming progress rather than replaying previously consumed Kafka offsets into the output.

### Stateful duplicate verification

I then published an additional retry of `STREAM_A100` and reran the streaming application.

The trusted event population remained **4**, demonstrating stateful event-ID deduplication within the configured watermark horizon.

## Conclusions and limitations

I implemented an isolated event-driven pipeline using Apache Kafka and Spark Structured Streaming.

Kafka provided a partitioned event log, while Spark consumed the topic, parsed JSON events, enforced the event contract, quarantined invalid records, and deduplicated valid events using event-time state.

I preserved Kafka partition and offset metadata in the output so processed events remain traceable to their source positions.

Spark checkpoints prevented already-consumed offsets from being written again when the application restarted, while a 10-minute watermark bounded the state used for event-ID deduplication.

### Limitations

This is a local single-broker development environment rather than a production Kafka cluster.

The synthetic events are not connected to the original X5 data and do not update Snowflake customer marts or machine-learning features.

The output currently uses local Parquet storage. A production architecture would use durable shared storage or another managed sink.

The demonstration uses one Spark process on a local machine. It does not test cluster-level scaling, broker replication, failure of individual Spark executors, schema-registry integration, or high-volume backpressure.

The next stage is to add observability and operational monitoring so the platform can detect data-quality failures, pipeline failures, and model-serving health issues rather than relying on manual inspection.